In [813]:
import pandas as pd 
import numpy as np
import re
from pathlib import Path

import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import display

import utils



In [814]:
all_ccf_df = pd.read_csv('all_ccf_df.csv')

In [815]:
all_ccf_df.columns

Index(['star_from_file', 'rjd', 'vrad', 'svrad', 'fwhm', 'sig_fwhm',
       'bis_span', 'sig_bis_span', 'contrast', 'sig_contrast', 's_mw', 'sig_s',
       'ha', 'sig_ha', 'na', 'sig_na', 'ca', 'sig_ca', 'rhk', 'sig_rhk',
       'berv', 'weight', 'file_rootpath', 'observation_file', 'true_vrad1',
       'true_vrad2', 'observation_name', 'ccf_fits_path', 'ccf_file_name',
       'iccf_bis', 'iccf_bis_error', 'iccf_fwhm', 'iccf_fwhm_error',
       'iccf_object', 'iccf_rv', 'iccf_rv_error', 'iccf_vspan', 'iccf_wspan',
       'iccf_ccf', 'iccf_rv_grid', 'iccf_contrast', 'iccf_contrast_error',
       'star', 'Name', 'gaia_dr3', 'Teff', 'eTeff', 'Logg', 'eLogg', '[Fe/H]',
       'e[Fe/H]', 'Vt', 'eVt', 'Gmag', 'Plx', 'Distance', 'Mass_t', 'Radius_t',
       'SWFlag', 'Reference', 'Link', 'Update', 'Database'],
      dtype='object')

In [816]:
all_ccf_df.head(2)

,star_from_file,rjd,vrad,svrad,fwhm,sig_fwhm,bis_span,sig_bis_span,contrast,sig_contrast,...,Gmag,Plx,Distance,Mass_t,Radius_t,SWFlag,Reference,Link,Update,Database
0,HD189733,59437.538315,-2174.760250,0.323563,7477.005863,0.647126,-32.279851,0.647126,57.246647,0.004955,...,7.4284,50.5668,19.775821,0.751097,0.786858,1,Sousa et al. 2021,https://ui.adsabs.harvard.edu/abs/2021arXiv210...,2021-01-01,EU;NASA
1,HD189733,59437.542268,-2177.035396,0.295833,7475.083070,0.591666,-31.580881,0.591666,57.238499,0.004531,...,7.4284,50.5668,19.775821,0.751097,0.786858,1,Sousa et al. 2021,https://ui.adsabs.harvard.edu/abs/2021arXiv210...,2021-01-01,EU;NASA


In [817]:
normalization_requests = [
    {
        "columns": ["iccf_rv", "iccf_fwhm", "iccf_bis", "iccf_vspan", "iccf_wspan", 'iccf_contrast'],
        "methods": ["subtract_mean", "fractional_mean", "zscore"],
        "group_column": "observation_file",
        "add_group_stats": True,
        "stat_columns": ["iccf_rv", "iccf_fwhm", "iccf_bis", "iccf_vspan", "iccf_wspan", 'iccf_contrast'],
    },
    {
        "columns": ["s_mw", "ha", "na", "ca", "rhk"],
        "methods": ["subtract_mean", "fractional_mean", "zscore"],
        "group_column": "observation_file",
        "add_group_stats": True,
        "stat_columns": ["s_mw", "ha", "na", "ca", "rhk"],
    },
]

all_ccf_df = utils.add_normalizations_from_requests(
    all_ccf_df,
    requests=normalization_requests,
)


In [818]:
"""
sample_catalog_df = load_sample_catalog("sample_sweetcat.csv")

extract_spectroscopy_archives(
    download_root="downloads/dace_ccf_a",
    extract_root="downloads/dace_ccf_a_extracted",
)

ccf_df = build_merged_ccf_table_for_observation(
    observation_name="HD189733_esp19_3.rdb",
    extracted_root="downloads/dace_ccf_a_extracted",
    observation_root="observations/Best_RM",
    corrected_root="observations/Best_RM/linear_corrected_with_uncertainties",
    sample_catalog_df=sample_catalog_df,
)

ccf_df.head()
"""

'\nsample_catalog_df = load_sample_catalog("sample_sweetcat.csv")\n\nextract_spectroscopy_archives(\n    download_root="downloads/dace_ccf_a",\n    extract_root="downloads/dace_ccf_a_extracted",\n)\n\nccf_df = build_merged_ccf_table_for_observation(\n    observation_name="HD189733_esp19_3.rdb",\n    extracted_root="downloads/dace_ccf_a_extracted",\n    observation_root="observations/Best_RM",\n    corrected_root="observations/Best_RM/linear_corrected_with_uncertainties",\n    sample_catalog_df=sample_catalog_df,\n)\n\nccf_df.head()\n'

In [819]:
rm_df = all_ccf_df

In [820]:
def plot_corrected_observation(obs_name, target_column):
    obs_df = rm_df.loc[rm_df['observation_file'] == obs_name].sort_values('rjd')

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=obs_df['rjd'],
            y=obs_df[target_column],
            mode='markers+lines',
            marker=dict(size=7),
            name=target_column,
            hovertemplate='rjd=%{x}<br>true_vrad=%{y}<extra></extra>',
        )
    )
    fig.update_layout(
        title=f'Linear-trend corrected RV: {obs_name}',
        xaxis_title='rjd',
        yaxis_title=target_column,
        template='plotly_white',
        hovermode='closest',
    )

    fig.show()


obs_selector = widgets.Dropdown(
    options=sorted(rm_df['observation_file'].unique()),
    description='obs_name:',
)
interactive_plot = widgets.interactive_output(plot_corrected_observation, {'obs_name': obs_selector},)
#display(obs_selector, interactive_plot)

In [821]:
full_feature_columns = [
 'iccf_fwhm_mean',
 'iccf_fwhm_std',
 'iccf_bis_mean',
 'iccf_bis_std',
 'iccf_vspan_mean',
 'iccf_vspan_std',
 'iccf_wspan_mean',
 'iccf_wspan_std',
 'iccf_contrast_mean',
 'iccf_contrast_std',
 'iccf_rv_subtract_mean',
 'iccf_fwhm_subtract_mean',
 'iccf_bis_subtract_mean',
 'iccf_vspan_subtract_mean',
 'iccf_wspan_subtract_mean',
 'iccf_contrast_subtract_mean',
 'iccf_fwhm_fractional_mean',
 'iccf_bis_fractional_mean',
 'iccf_vspan_fractional_mean',
 'iccf_wspan_fractional_mean',
 'iccf_contrast_fractional_mean',
 'iccf_fwhm_zscore',
 'iccf_bis_zscore',
 'iccf_vspan_zscore',
 'iccf_wspan_zscore',
 'iccf_contrast_zscore',
 's_mw_mean',
 's_mw_std',
 'ha_mean',
 'ha_std',
 'ca_mean',
 'ca_std',
 's_mw_subtract_mean',
 'ha_subtract_mean',
 'ca_subtract_mean',
 's_mw_fractional_mean',
 'ha_fractional_mean',
 'ca_fractional_mean',
 's_mw_zscore',
 'ha_zscore',
 'ca_zscore',
]

subtract_mean_features = ['iccf_rv_subtract_mean',
 'iccf_fwhm_subtract_mean',
 'iccf_bis_subtract_mean',
 'iccf_vspan_subtract_mean',
 'iccf_wspan_subtract_mean',
 'iccf_contrast_subtract_mean',]


subtract_fractional_mean_features = ['iccf_rv_subtract_mean',
 'iccf_fwhm_fractional_mean',
 'iccf_bis_fractional_mean',
 'iccf_vspan_fractional_mean',
 'iccf_wspan_fractional_mean',
 'iccf_contrast_fractional_mean',]

zscore_features = ['iccf_rv_subtract_mean',
 'iccf_fwhm_zscore',
 'iccf_bis_zscore',
 'iccf_vspan_zscore',
 'iccf_wspan_zscore',
 'iccf_contrast_zscore',]

In [822]:
test_stars = ['K2139', 'WASP126', 'WASP166']
train_rm_df = rm_df.loc[~rm_df['star_from_file'].isin(test_stars)].copy()
test_rm_df = rm_df.loc[rm_df['star_from_file'].isin(test_stars)].copy()
test_label = ', '.join(test_stars)

display(
    pd.Series(
        {
            'n_train_rows': len(train_rm_df),
            'n_test_rows': len(test_rm_df),
            'n_test_observations': test_rm_df['observation_file'].nunique(),
        }
    ).to_frame('value')
)

,value
n_train_rows,913
n_test_rows,258
n_test_observations,3


In [ ]:
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, StackingRegressor, VotingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, KFold
from sklearn.neighbors import KNeighborsRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

activity_feature_columns = ['iccf_contrast_mean','iccf_rv_subtract_mean',   'ha_zscore',
                            'iccf_wspan_fractional_mean', 'iccf_bis_zscore', 'iccf_contrast_zscore',
                             'iccf_wspan_zscore',]

optional_feature_columns = []

target_column = 'true_vrad2'

feature_sets = {'activity_only': activity_feature_columns}
for optional_column in optional_feature_columns:
    feature_sets[f'activity_plus_{optional_column}'] = activity_feature_columns + [optional_column]
feature_sets['activity_plus_all_optional'] = activity_feature_columns + optional_feature_columns


def make_linear_model():
    return LinearRegression()


def make_ridge_model():
    return Ridge(alpha=1.0)


def make_knn_model():
    return KNeighborsRegressor(n_neighbors=15, weights='distance')


def make_random_forest_model():
    return RandomForestRegressor(
        random_state=42,
        n_jobs=-1,
    )


def make_extra_trees_model():
    return ExtraTreesRegressor(
        random_state=42,
        n_jobs=-1,
    )


def make_xgb_model():
    return XGBRegressor(
        random_state=42,
    )


def make_catboost_model():
    return CatBoostRegressor(
        random_seed=42,
        verbose=0,
        allow_writing_files=False,
    )


def make_lightgbm_model():
    return LGBMRegressor(
        random_state=42,
        verbosity=-1,
    )


def build_stack_cv_splits(train_df, feature_columns):
    n_group_splits = min(5, train_df['observation_file'].nunique())
    if n_group_splits >= 2:
        splitter = GroupKFold(n_splits=n_group_splits)
        return list(
            splitter.split(
                train_df[feature_columns],
                train_df[target_column],
                train_df['observation_file'],
            )
        )

    n_row_splits = min(5, len(train_df))
    if n_row_splits < 2:
        raise ValueError('Need at least 2 rows to build stacking folds.')
    splitter = KFold(n_splits=n_row_splits, shuffle=True, random_state=42)
    return list(splitter.split(train_df[feature_columns], train_df[target_column]))


def build_model_factories(stack_cv_splits):
    def make_vote_linear_tree_model():
        return VotingRegressor(
            estimators=[
                ('ridge', make_ridge_model()),
                ('rf', make_random_forest_model()),
                ('xgb', make_xgb_model()),
            ]
        )

    def make_vote_tree_model():
        return VotingRegressor(
            estimators=[
                ('rf', make_random_forest_model()),
                ('et', make_extra_trees_model()),
                ('xgb', make_xgb_model()),
            ]
        )

    def make_vote_all_model():
        return VotingRegressor(
            estimators=[
                ('ridge', make_ridge_model()),
                ('knn', make_knn_model()),
                ('rf', make_random_forest_model()),
                ('et', make_extra_trees_model()),
                ('xgb', make_xgb_model()),
            ]
        )

    def make_stack_linear_tree_model():
        return StackingRegressor(
            estimators=[
                ('ridge', make_ridge_model()),
                ('rf', make_random_forest_model()),
                ('xgb', make_xgb_model()),
            ],
            final_estimator=Ridge(alpha=1.0),
            passthrough=True,
            cv=stack_cv_splits,
            n_jobs=-1,
        )

    def make_stack_tree_boost_model():
        return StackingRegressor(
            estimators=[
                ('rf', make_random_forest_model()),
                ('et', make_extra_trees_model()),
                ('xgb', make_xgb_model()),
                ('catboost', make_catboost_model()),
                ('lightgbm', make_lightgbm_model()),
            ],
            final_estimator=Ridge(alpha=1.0),
            passthrough=True,
            cv=stack_cv_splits,
            n_jobs=-1,
        )

    def make_stack_all_model():
        return StackingRegressor(
            estimators=[
                ('ridge', make_ridge_model()),
                ('knn', make_knn_model()),
                ('rf', make_random_forest_model()),
                ('et', make_extra_trees_model()),
                ('xgb', make_xgb_model()),
                ('catboost', make_catboost_model()),
                ('lightgbm', make_lightgbm_model()),
            ],
            final_estimator=Ridge(alpha=0.5),
            passthrough=True,
            cv=stack_cv_splits,
            n_jobs=-1,
        )

    return {
        'linear': make_linear_model,
        'ridge': make_ridge_model,
        'knn': make_knn_model,
        'random_forest': make_random_forest_model,
        'extra_trees': make_extra_trees_model,
        'xgboost': make_xgb_model,
        'catboost': make_catboost_model,
        'lightgbm': make_lightgbm_model,
        'vote_linear_tree': make_vote_linear_tree_model,
        'vote_tree': make_vote_tree_model,
        'vote_all': make_vote_all_model,
        'stack_linear_tree': make_stack_linear_tree_model,
        'stack_tree_boost': make_stack_tree_boost_model,
        'stack_all': make_stack_all_model,
    }


xgb_models = {}
prediction_tables = {}
metric_rows = []

for feature_set_name, raw_feature_columns in feature_sets.items():
    feature_columns = []
    seen_columns = set()
    for column in raw_feature_columns:
        if column in rm_df.columns and column not in seen_columns:
            feature_columns.append(column)
            seen_columns.add(column)

    required_columns = [*feature_columns, target_column]
    train_df = train_rm_df.dropna(subset=required_columns).copy()
    test_df = test_rm_df.dropna(subset=required_columns).copy()

    if train_df.empty:
        raise ValueError(f'No training rows remain for feature set {feature_set_name!r}.')
    if test_df.empty:
        raise ValueError(f'No test rows remain for feature set {feature_set_name!r}.')

    stack_cv_splits = build_stack_cv_splits(train_df, feature_columns)
    model_factories = build_model_factories(stack_cv_splits)

    for estimator_name, model_factory in model_factories.items():
        model_name = f'{estimator_name}__{feature_set_name}'
        model = model_factory()
        model.fit(train_df[feature_columns], train_df[target_column])

        prediction_df = test_df.sort_values(['observation_file', 'rjd']).copy()
        prediction_df['predicted_true_vrad'] = model.predict(prediction_df[feature_columns])
        prediction_df['model_name'] = model_name
        prediction_df['estimator_name'] = estimator_name
        prediction_df['feature_set_name'] = feature_set_name

        test_mae = mean_absolute_error(
            prediction_df[target_column],
            prediction_df['predicted_true_vrad'],
        )
        test_rmse = mean_squared_error(
            prediction_df[target_column],
            prediction_df['predicted_true_vrad'],
        ) ** 0.5
        test_r2 = r2_score(
            prediction_df[target_column],
            prediction_df['predicted_true_vrad'],
        )

        metric_rows.append(
            {
                'model_name': model_name,
                'estimator_name': estimator_name,
                'feature_set_name': feature_set_name,
                'n_features': len(feature_columns),
                'n_train_rows': len(train_df),
                'n_test_rows': len(prediction_df),
                'n_stack_folds': len(stack_cv_splits) if estimator_name.startswith('stack_') else None,
                'test_mae': test_mae,
                'test_rmse': test_rmse,
                'test_r2': test_r2,
                'features': feature_columns,
            }
        )

        xgb_models[model_name] = model
        prediction_tables[model_name] = prediction_df

xgb_results_df = pd.DataFrame(metric_rows).sort_values(
    ['test_rmse', 'test_mae', 'n_features']
).reset_index(drop=True)

display(
    xgb_results_df[
        [
            'model_name',
            'estimator_name',
            'feature_set_name',
            'n_features',
            'n_train_rows',
            'n_test_rows',
            'n_stack_folds',
            'test_mae',
            'test_rmse',
            'test_r2',
            'features',
        ]
    ]
)

display(
    xgb_results_df.loc[
        xgb_results_df['estimator_name'].isin(['catboost', 'lightgbm']),
        [
            'model_name',
            'estimator_name',
            'feature_set_name',
            'test_mae',
            'test_rmse',
            'test_r2',
        ],
    ].reset_index(drop=True)
)

best_by_estimator_df = (
    xgb_results_df.sort_values(['test_rmse', 'test_mae'])
    .groupby('estimator_name', as_index=False)
    .first()[
        [
            'estimator_name',
            'model_name',
            'feature_set_name',
            'test_mae',
            'test_rmse',
            'test_r2',
        ]
    ]
    .sort_values(['test_rmse', 'test_mae'])
    .reset_index(drop=True)
)
#display(best_by_estimator_df)

best_model_name = xgb_results_df.loc[0, 'model_name']
prediction_df = prediction_tables[best_model_name]
print(f'Best model by test RMSE: {best_model_name}')


,model_name,estimator_name,feature_set_name,n_features,n_train_rows,n_test_rows,n_stack_folds,test_mae,test_rmse,test_r2,features
0,vote_linear_tree__activity_only,vote_linear_tree,activity_only,7,913,258,NaN,2.750955,3.441354,-2.730303,"[iccf_contrast_mean, iccf_rv_subtract_mean, ha..."
1,vote_linear_tree__activity_plus_all_optional,vote_linear_tree,activity_plus_all_optional,7,913,258,NaN,2.750955,3.441354,-2.730303,"[iccf_contrast_mean, iccf_rv_subtract_mean, ha..."
2,xgboost__activity_only,xgboost,activity_only,7,913,258,NaN,2.953901,3.560340,-2.992715,"[iccf_contrast_mean, iccf_rv_subtract_mean, ha..."
3,xgboost__activity_plus_all_optional,xgboost,activity_plus_all_optional,7,913,258,NaN,2.953901,3.560340,-2.992715,"[iccf_contrast_mean, iccf_rv_subtract_mean, ha..."
4,vote_all__activity_only,vote_all,activity_only,7,913,258,NaN,3.012372,3.750496,-3.430602,"[iccf_contrast_mean, iccf_rv_subtract_mean, ha..."
5,vote_all__activity_plus_all_optional,vote_all,activity_plus_all_optional,7,913,258,NaN,3.012372,3.750496,-3.430602,"[iccf_contrast_mean, iccf_rv_subtract_mean, ha..."
6,vote_tree__activity_only,vote_tree,activity_only,7,913,258,NaN,3.093760,3.762956,-3.460090,"[iccf_contrast_mean, iccf_rv_subtract_mean, ha..."
7,vote_tree__activity_plus_all_optional,vote_tree,activity_plus_all_optional,7,913,258,NaN,3.093760,3.762956,-3.460090,"[iccf_contrast_mean, iccf_rv_subtract_mean, ha..."
8,stack_linear_tree__activity_only,stack_linear_tree,activity_only,7,913,258,5.0,3.185625,3.904919,-3.802965,"[iccf_contrast_mean, iccf_rv_subtract_mean, ha..."
9,stack_linear_tree__activity_plus_all_optional,stack_linear_tree,activity_plus_all_optional,7,913,258,5.0,3.185625,3.904919,-3.802965,"[iccf_contrast_mean, iccf_rv_subtract_mean, ha..."


,model_name,estimator_name,feature_set_name,test_mae,test_rmse,test_r2
0,lightgbm__activity_only,lightgbm,activity_only,3.504027,4.343234,-4.941713
1,lightgbm__activity_plus_all_optional,lightgbm,activity_plus_all_optional,3.504027,4.343234,-4.941713
2,catboost__activity_only,catboost,activity_only,3.886162,5.326376,-7.936119
3,catboost__activity_plus_all_optional,catboost,activity_plus_all_optional,3.886162,5.326376,-7.936119


,estimator_name,model_name,feature_set_name,test_mae,test_rmse,test_r2
0,vote_linear_tree,vote_linear_tree__activity_only,activity_only,2.750955,3.441354,-2.730303
1,xgboost,xgboost__activity_only,activity_only,2.953901,3.560340,-2.992715
2,vote_all,vote_all__activity_only,activity_only,3.012372,3.750496,-3.430602
3,vote_tree,vote_tree__activity_only,activity_only,3.093760,3.762956,-3.460090
4,stack_linear_tree,stack_linear_tree__activity_only,activity_only,3.185625,3.904919,-3.802965
5,stack_all,stack_all__activity_only,activity_only,3.218893,4.014699,-4.076813
6,random_forest,random_forest__activity_only,activity_only,3.328024,4.042341,-4.146965
7,extra_trees,extra_trees__activity_only,activity_only,3.340901,4.190464,-4.531074
8,lightgbm,lightgbm__activity_only,activity_only,3.504027,4.343234,-4.941713
9,stack_tree_boost,stack_tree_boost__activity_only,activity_only,3.657230,4.604588,-5.678315


Best model by test RMSE: vote_linear_tree__activity_only


In [824]:
model_to_plot = best_model_name
prediction_df = prediction_tables[model_to_plot]
plot_metrics = xgb_results_df.set_index("model_name").loc[model_to_plot]

display(
    pd.Series(
        {
            "model_name": model_to_plot,
            "n_train_rows": plot_metrics["n_train_rows"],
            "test_mae": plot_metrics["test_mae"],
            "test_rmse": plot_metrics["test_rmse"],
            "test_r2": plot_metrics["test_r2"],
            "n_test_rows": len(prediction_df),
        }
    ).to_frame("value")
)


,value
model_name,vote_linear_tree__activity_only
n_train_rows,913
test_mae,2.750955
test_rmse,3.441354
test_r2,-2.730303
n_test_rows,258


In [825]:
model_to_plot = best_model_name#best_model_name
prediction_df = prediction_tables[model_to_plot]
plot_metrics = xgb_results_df.set_index('model_name').loc[model_to_plot]

rjd_min = 60.1355e3
rjd_max = 60.1359e3
plot_prediction_df = prediction_df.loc[
    prediction_df['rjd'].between(rjd_min, rjd_max)
].copy()

if plot_prediction_df.empty:
    raise ValueError(f'No prediction rows found in rjd range [{rjd_min}, {rjd_max}].')

display(
    pd.Series(
    {
        'model_name': model_to_plot,
        'mae': plot_metrics['test_mae'],
        'rmse': plot_metrics['test_rmse'],
        'r2': plot_metrics['test_r2'],
        'n_test_rows': len(prediction_df),
        'n_plot_rows': len(plot_prediction_df),
        'rjd_min': rjd_min,
        'rjd_max': rjd_max,
    }
    ).to_frame('value')
)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=plot_prediction_df['rjd'],
        y=plot_prediction_df['iccf_rv_subtract_mean'],
        mode='markers+lines',
        marker=dict(size=7),
        name='vrad',
        hovertemplate='rjd=%{x}<br>vrad=%{y}<extra></extra>',
    )
)
fig.add_trace(
    go.Scatter(
        x=plot_prediction_df['rjd'],
        y=plot_prediction_df['predicted_true_vrad'],
        mode='markers+lines',
        marker=dict(size=7),
        name='predicted true_vrad',
        hovertemplate='rjd=%{x}<br>predicted true_vrad=%{y}<extra></extra>',
    )
)
fig.add_trace(
    go.Scatter(
        x=plot_prediction_df['rjd'],
        y=plot_prediction_df[target_column],
        mode='lines',
        line=dict(width=2, dash='dash'),
        name='actual true_vrad',
        hovertemplate='rjd=%{x}<br>actual true_vrad=%{y}<extra></extra>',
    )
)
fig.update_layout(
    title=f'XGBoost test prediction: {test_label} ({model_to_plot}) | rjd [{rjd_min}, {rjd_max}]',
    xaxis_title='rjd',
    yaxis_title='RV',
    template='plotly_white',
    hovermode='closest',
)
fig.show()

plot_prediction_df.head()

,value
model_name,vote_linear_tree__activity_only
mae,2.750955
rmse,3.441354
r2,-2.730303
n_test_rows,258
n_plot_rows,60
rjd_min,60135.5
rjd_max,60135.9


,star_from_file,rjd,vrad,svrad,fwhm,sig_fwhm,bis_span,sig_bis_span,contrast,sig_contrast,...,rhk_fractional_mean,s_mw_zscore,ha_zscore,na_zscore,ca_zscore,rhk_zscore,predicted_true_vrad,model_name,estimator_name,feature_set_name
357,K2139,60135.543241,-31338.766939,3.774139,7263.473084,7.548278,-25.617252,7.548278,61.190992,0.063590,...,0.000268,-0.077383,1.399988,-0.031625,-0.077410,-0.044100,-1.701925,vote_linear_tree__activity_only,vote_linear_tree,activity_only
358,K2139,60135.548898,-31335.896873,3.383965,7263.081180,6.767929,-32.704594,6.767929,61.143020,0.056975,...,0.002614,-0.475288,-1.449256,-1.206886,-0.475307,-0.430280,-1.961886,vote_linear_tree__activity_only,vote_linear_tree,activity_only
359,K2139,60135.554669,-31343.657911,4.292692,7270.867877,8.585383,-38.122918,8.585383,60.946776,0.071965,...,0.002859,-0.516246,-0.649066,-1.720686,-0.516253,-0.470564,-4.655161,vote_linear_tree__activity_only,vote_linear_tree,activity_only
360,K2139,60135.560812,-31332.207655,3.104769,7277.738035,6.209538,-23.467984,6.209538,61.140262,0.052166,...,-0.002520,0.408268,0.464873,-0.317590,0.408268,0.414885,1.786159,vote_linear_tree__activity_only,vote_linear_tree,activity_only
361,K2139,60135.566622,-31338.623985,2.295245,7280.459118,4.590490,-33.799995,4.590490,61.140779,0.038551,...,0.000577,-0.130411,-1.803676,1.107555,-0.130417,-0.095004,-1.767573,vote_linear_tree__activity_only,vote_linear_tree,activity_only


In [826]:
star_ccf = pd.read_csv("observations/test_observations/K2-111_ccf_table.csv")

star_df = utils.add_normalizations_from_requests(
    star_ccf,
    requests=normalization_requests,
)
#star_df

In [827]:
model_to_apply = best_model_name
model = xgb_models[model_to_apply]
feature_columns = list(xgb_results_df.set_index('model_name').loc[model_to_apply, 'features'])

missing_feature_columns = [column for column in feature_columns if column not in star_df.columns]
if missing_feature_columns:
    raise KeyError(f'Missing derived feature columns for prediction: {missing_feature_columns}')

star_prediction_df = star_df.copy()
for column in feature_columns:
    star_prediction_df[column] = pd.to_numeric(star_prediction_df[column], errors='coerce')

star_prediction_df = star_prediction_df.dropna(subset=feature_columns).sort_values(
    ['observation_file', 'rjd']
).copy()
star_prediction_df['predicted_true_vrad'] = model.predict(star_prediction_df[feature_columns])

display(
    pd.Series(
        {
            'model_name': model_to_apply,
            'estimator_name': xgb_results_df.set_index('model_name').loc[model_to_apply, 'estimator_name'],
            'feature_set_name': xgb_results_df.set_index('model_name').loc[model_to_apply, 'feature_set_name'],
            'features': feature_columns,
            'n_star_rows': len(star_prediction_df),
            'n_star_observations': star_prediction_df['observation_file'].nunique(),
        }
    ).to_frame('value')
)

fig = go.Figure()
for obs_name, obs_df in star_prediction_df.groupby('observation_file', sort=True):
    fig.add_trace(
        go.Scatter(
            x=obs_df['rjd'],
            y=obs_df['iccf_rv_subtract_mean'],
            mode='markers+lines',
            marker=dict(size=7),
            name=f'{obs_name} vrad',
            hovertemplate='obs=%{fullData.name}<br>rjd=%{x}<br>vrad=%{y}<extra></extra>',
        )
    )
    fig.add_trace(
        go.Scatter(
            x=obs_df['rjd'],
            y=obs_df['predicted_true_vrad'],
            mode='markers+lines',
            marker=dict(size=7, symbol='diamond'),
            name=f'{obs_name} predicted true_vrad',
            hovertemplate='obs=%{fullData.name}<br>rjd=%{x}<br>predicted true_vrad=%{y}<extra></extra>',
        )
    )
fig.update_layout(
    title=f'Model prediction for star ({model_to_apply})',
    xaxis_title='rjd',
    yaxis_title='RV',
    template='plotly_white',
    hovermode='closest',
)
fig.show()

star_prediction_df[['observation_file', 'rjd', 'vrad', 'predicted_true_vrad', *feature_columns]].head()


,value
model_name,vote_linear_tree__activity_only
estimator_name,vote_linear_tree
feature_set_name,activity_only
features,"[iccf_contrast_mean, iccf_rv_subtract_mean, ha..."
n_star_rows,41
n_star_observations,1


,observation_file,rjd,vrad,predicted_true_vrad,iccf_contrast_mean,iccf_rv_subtract_mean,ha_zscore,iccf_wspan_fractional_mean,iccf_bis_zscore,iccf_contrast_zscore,iccf_wspan_zscore
0,K2-111_ESPRESSO19_1.rdb,58421.739869,-16433.631324,-1.860942,41.276015,-1.853059,-1.038165,-0.038767,0.487502,0.282663,0.138774
1,K2-111_ESPRESSO19_1.rdb,58424.779511,-16430.688374,2.859287,41.276015,1.091887,0.339453,0.226692,0.469669,0.508890,-0.811482
2,K2-111_ESPRESSO19_1.rdb,58426.757724,-16426.812419,6.208320,41.276015,4.967620,-0.336697,-0.345585,1.098306,0.715885,1.237079
3,K2-111_ESPRESSO19_1.rdb,58439.593202,-16436.001361,-3.262229,41.276015,-4.218171,-3.802047,0.446593,-1.210165,1.394816,-1.598653
4,K2-111_ESPRESSO19_1.rdb,58459.633960,-16426.961383,5.839655,41.276015,4.824569,0.512350,0.187507,0.515739,1.094893,-0.671213


In [828]:
star_prediction_df[['iccf_rv_subtract_mean', 'predicted_true_vrad']].describe()

,iccf_rv_subtract_mean,predicted_true_vrad
count,4.100000e+01,41.000000
mean,1.064774e-12,0.211136
std,4.305081e+00,4.310317
min,-1.185329e+01,-7.808843
25%,-2.612184e+00,-3.108837
50%,-3.939373e-01,0.358607
75%,3.149916e+00,3.671517
max,7.481505e+00,9.086055


In [829]:
from astropy.timeseries import LombScargle

gls_input_df = (
    star_prediction_df[['rjd', 'predicted_true_vrad']]
    .dropna()
    .sort_values('rjd')
    .copy()
)

gls = LombScargle(
    gls_input_df['rjd'].to_numpy(),
    gls_input_df['predicted_true_vrad'].to_numpy(),
    center_data=True,
    fit_mean=True,
)

min_period = 1
max_period = 200

min_frequency = 1 / max_period
max_frequency = 1 / min_period

frequency, power = gls.autopower(
    minimum_frequency=min_frequency,
    maximum_frequency=max_frequency,
)

#frequency, power = gls.autopower()


best_frequency = frequency[np.argmax(power)]
best_period = 1 / best_frequency

gls_periodogram_df = pd.DataFrame(
    {
        'frequency': frequency,
        'period': 1 / frequency,
        'power': power,
    }
).sort_values('period')

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=gls_periodogram_df['period'],
        y=gls_periodogram_df['power'],
        mode='lines',
        name='GLS power',
        hovertemplate='period=%{x:.6f} d<br>power=%{y:.6f}<extra></extra>',
    )
)
fig.add_vline(
    x=best_period,
    line_dash='dash',
    line_color='crimson',
    annotation_text=f'best period = {best_period:.6f} d',
    annotation_position='top right',
)
fig.update_layout(
    title='GLS periodogram of predicted RV',
    xaxis_title='Period [days]',
    yaxis_title='Power',
)
fig.update_xaxes(type='log', range=[np.log10(1), np.log10(200)])
fig.show()

pd.Series(
    {
        'best_frequency': best_frequency,
        'best_period_days': best_period,
        'max_power': power.max(),
    }
)


best_frequency      0.913606
best_period_days    1.094564
max_power           0.406676
dtype: float64

In [830]:
target_window_min = 5.05
target_window_max = 5.15
background_window_min = 6.0
background_window_max = 8.0
eps = 1e-12

target_mask = (
    (gls_periodogram_df['period'] >= target_window_min)
    & (gls_periodogram_df['period'] <= target_window_max)
)

background_mask = (
    (gls_periodogram_df['period'] >= background_window_min)
    & (gls_periodogram_df['period'] <= background_window_max)
)

target_window_df = gls_periodogram_df.loc[target_mask].copy()
background_window_df = gls_periodogram_df.loc[background_mask].copy()

if target_window_df.empty:
    raise ValueError('No GLS samples found in target window.')
if background_window_df.empty:
    raise ValueError('No GLS samples found in background window.')

peak_idx = target_window_df['power'].idxmax()
peak_period = gls_periodogram_df.loc[peak_idx, 'period']
peak_power = gls_periodogram_df.loc[peak_idx, 'power']
background_median_power = background_window_df['power'].median()

gls_ratio = peak_power / max(background_median_power, eps)

pd.Series(
    {
        'target_window_min': target_window_min,
        'target_window_max': target_window_max,
        'background_window_min': background_window_min,
        'background_window_max': background_window_max,
        'peak_period_in_target_window': peak_period,
        'peak_power_in_target_window': peak_power,
        'median_power_in_background_window': background_median_power,
        'gls_ratio': gls_ratio,
    }
).to_frame('value')


,value
target_window_min,5.050000
target_window_max,5.150000
background_window_min,6.000000
background_window_max,8.000000
peak_period_in_target_window,5.067433
peak_power_in_target_window,0.062815
median_power_in_background_window,0.028144
gls_ratio,2.231918


In [831]:
import matplotlib.pyplot as plt     
#plt.scatter(star_prediction_df['contrast'], star_prediction_df['predicted_true_vrad'] - star_prediction_df['vrad_subtract_mean'])

In [832]:
rm_df.columns

Index(['star_from_file', 'rjd', 'vrad', 'svrad', 'fwhm', 'sig_fwhm',
       'bis_span', 'sig_bis_span', 'contrast', 'sig_contrast',
       ...
       's_mw_fractional_mean', 'ha_fractional_mean', 'na_fractional_mean',
       'ca_fractional_mean', 'rhk_fractional_mean', 's_mw_zscore', 'ha_zscore',
       'na_zscore', 'ca_zscore', 'rhk_zscore'],
      dtype='object', length=118)

In [833]:
export_dir = Path('model_tables')
export_dir.mkdir(exist_ok=True)

rm_export_path = export_dir / 'rm_df_full.csv'
rm_df.to_csv(rm_export_path, index=False)

star_export_df = star_df.reindex(columns=rm_df.columns)
star_export_path = export_dir / 'hd22496_full.csv'
star_export_df.to_csv(star_export_path, index=False)

pd.Series(
    {
        'rm_export_path': str(rm_export_path),
        'rm_n_rows': len(rm_df),
        'rm_n_columns': len(rm_df.columns),
        'star_export_path': str(star_export_path),
        'star_n_rows': len(star_export_df),
        'star_n_columns': len(star_export_df.columns),
    }
).to_frame('value')


,value
rm_export_path,model_tables/rm_df_full.csv
rm_n_rows,1171
rm_n_columns,118
star_export_path,model_tables/hd22496_full.csv
star_n_rows,41
star_n_columns,118


In [834]:
to_save = star_prediction_df[['rjd', 'predicted_true_vrad']].copy()
to_save = to_save.rename(columns={'predicted_true_vrad': 'vrad'})
to_save['svrad'] = 0.2

output_path = Path('hd22496_prediction.rdb')
with output_path.open('w', encoding='utf-8') as f:
    f.write('rjd\tvrad\tsvrad\n')
    f.write('---\t---\t---\n')
    to_save.to_csv(f, index=False, sep='\t', header=False)

output_path

PosixPath('hd22496_prediction.rdb')

In [835]:
to_save

,rjd,vrad,svrad
0,58421.739869,-1.860942,0.2
1,58424.779511,2.859287,0.2
2,58426.757724,6.208320,0.2
3,58439.593202,-3.262229,0.2
4,58459.633960,5.839655,0.2
5,58463.730818,5.553211,0.2
6,58466.628217,0.618412,0.2
7,58470.700200,9.086055,0.2
8,58488.635526,-4.207306,0.2
9,58493.592841,3.671517,0.2
